# PathSnap training proof

**This notebook was executed locally with the project virtual environment.** Its saved output records an actual two-repeat, eight-fold training comparison of five learned models, a fresh Optuna search, and a refit of the selected trained artifact. The code also runs in Google Colab after uploading `PathSnap_Evaluator_Ready.zip`; Colab may take substantially longer. The final model choice uses pre-existing training CV evidence, not evaluator labels. The correction to use a trained default occurred after evaluator data became available, so this is not a pristine blind evaluator holdout.

In [1]:
import hashlib, json, os, subprocess, sys, time
from pathlib import Path
from zipfile import ZipFile
import joblib, pandas as pd
from IPython.display import display

archive = Path('/content/PathSnap_Evaluator_Ready.zip')
if archive.is_file():
    with ZipFile(archive) as z:
        z.extractall('/content/pathsnap_proof')
    project = Path('/content/pathsnap_proof/pathsnap')
else:
    project = Path.cwd().resolve()
raw = project / 'inputs/Student folder.zip'
assert raw.is_file(), f'Missing original training data: {raw}'
with ZipFile(raw) as z:
    names = z.namelist()
train_ids = sorted({n.split('/')[2] for n in names if n.startswith('Student folder/train/') and n.endswith('.track')})
test_ids = sorted({n.split('/')[2] for n in names if n.startswith('Student folder/student_test/') and n.endswith('.track')})
assert len(train_ids) == 80 and len(test_ids) == 10
print('Runtime Python:', sys.executable)
print('Raw dataset SHA256:', hashlib.sha256(raw.read_bytes()).hexdigest())
print('Training trips:', len(train_ids), 'Student-test trips:', len(test_ids))

def run(*args):
    start = time.time()
    print('RUN', sys.executable, *args, flush=True)
    proc = subprocess.run([sys.executable, '-u', *args], cwd=project, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          env=os.environ | {'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'4','PYTHONUNBUFFERED':'1'})
    print(proc.stdout, flush=True)
    print('Exit:', proc.returncode, 'Elapsed seconds:', round(time.time()-start, 1), flush=True)
    proc.check_returncode()

Runtime Python: /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/.venv/bin/python
Raw dataset SHA256: 4804b0399d1d56f4844b9c74e2f998aae0a281fb5947a1f47ad1bd013214d248
Training trips: 80 Student-test trips: 10


In [2]:
# Reuse a verified local feature cache if present; on Colab prepare from raw trip files.
local_cache = project.parent.parent / 'work/grouped'
prepared = local_cache if len(list((local_cache / 'train').glob('*.npz'))) == 80 else project / 'work/proof-prepared'
if len(list((prepared / 'train').glob('*.npz'))) != 80:
    run('-m', 'pathsnap.prepare_grouped', str(raw), str(prepared))
assert len(list((prepared / 'train').glob('*.npz'))) == 80
print('Prepared feature source:', prepared)
print('Cache trip files:', len(list((prepared / 'train').glob('*.npz'))))

Prepared feature source: /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/work/grouped
Cache trip files: 80


In [3]:
# Every model is fitted 16 times: two repeats of eight whole-trip folds.
five_out = project / 'results/notebook_proof/five_model'
run('-m', 'pathsnap.grouped_training', str(prepared), str(five_out),
    '--mode', 'compare', '--names', 'Conditional logit', 'Grouped MLP',
    'ExtraTrees', 'LightGBM ranker', 'HistGradientBoosting', '--repeats', '2')
frames = [pd.read_csv(p) for p in sorted(five_out.glob('folds_*.csv'))]
assert len(frames) == 5 and all(len(f) == 16 for f in frames)
five_summary = pd.concat(frames).groupby('model')[['top1','grouped_log_loss']].agg(['mean','std'])
print('FRESH five-model grouped-CV comparison')
display(five_summary)

RUN /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/.venv/bin/python -m pathsnap.grouped_training /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/work/grouped /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/results/notebook_proof/five_model --mode compare --names Conditional logit Grouped MLP ExtraTrees LightGBM ranker HistGradientBoosting --repeats 2


Conditional logit 1 1 top1=0.8299, nll=0.7263
Conditional logit 1 2 top1=0.8733, nll=0.5610
Conditional logit 1 3 top1=0.2768, nll=18.2650
Conditional logit 1 4 top1=0.8803, nll=0.5261
Conditional logit 1 5 top1=0.8781, nll=0.5195
Conditional logit 1 6 top1=0.7838, nll=1.7674
Conditional logit 1 7 top1=0.7531, nll=4.5457
Conditional logit 1 8 top1=0.8590, nll=0.9594
Conditional logit 2 1 top1=0.2961, nll=18.2095
Conditional logit 2 2 top1=0.8454, nll=0.5626
Conditional logit 2 3 top1=0.8364, nll=0.8102
Conditional logit 2 4 top1=0.8905, nll=0.5379
Conditional logit 2 5 top1=0.8995, nll=0.4807
Conditional logit 2 6 top1=0.8934, nll=0.8546
Conditional logit 2 7 top1=0.7751, nll=1.5257
Conditional logit 2 8 top1=0.7023, nll=4.7775
Grouped MLP 1 1 top1=0.8315, nll=0.6380
Grouped MLP 1 2 top1=0.8663, nll=0.5044
Grouped MLP 1 3 top1=0.2784, nll=18.2283
Grouped MLP 1 4 top1=0.8907, nll=0.4247
Grouped MLP 1 5 top1=0.8801, nll=0.4388
Grouped MLP 1 6 top1=0.7927, nll=1.6652
Grouped MLP 1 7 top1=

Exit: 0 Elapsed seconds: 322.9


FRESH five-model grouped-CV comparison


top1           grouped_log_loss          
                          mean       std             mean       std
model                                                              
Conditional logit     0.767067  0.195870         3.478572  5.916907
ExtraTrees            0.770490  0.196080         3.415047  5.922460
Grouped MLP           0.772757  0.195851         3.398405  5.931993
HistGradientBoosting  0.778848  0.199025         3.379941  5.931151
LightGBM ranker       0.779951  0.197694         3.373658  5.931969

In [4]:
# Bayesian hyperparameter search for the LightGBM ranking family.
local_context = project.parent.parent / 'work/review-data'
context = local_context if len(list((local_context / 'train').glob('*.npz'))) == 80 else project / 'work/proof-context'
if len(list((context / 'train').glob('*.npz'))) != 80:
    run('-m', 'pathsnap.prepare_grouped', str(raw), str(context), '--context')
ranker_out = project / 'results/notebook_proof/ranker'
run('-m', 'pathsnap.review_experiment', str(raw), str(context), str(ranker_out),
    '--phase', 'ranker', '--trials', '18')
trials = pd.read_csv(ranker_out / 'ranker_optuna_trials.csv')
assert len(trials) == 18
print('Best TPE parameters:', json.loads((ranker_out / 'ranker_parameters.json').read_text()))
display(trials[['number','value','state','params_n_estimators','params_num_leaves','params_max_depth']])

RUN /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/.venv/bin/python -m pathsnap.review_experiment /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/inputs/Student folder.zip /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/work/review-data /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/results/notebook_proof/ranker --phase ranker --trials 18


[I 2026-11-18 16:18:36,805] A new study created in memory with name: no-name-10e5af7c-8ace-43cf-baeb-d301b744d883
[I 2026-11-18 16:18:39,305] Trial 0 finished with value: 0.28185752712518414 and parameters: {'n_estimators': 90, 'num_leaves': 32, 'max_depth': 10, 'learning_rate': 0.08, 'min_child_samples': 40, 'reg_lambda': 0.01}. Best is trial 0 with value: 0.28185752712518414.
[I 2026-11-18 16:18:42,033] Trial 1 finished with value: 0.27588867665598266 and parameters: {'n_estimators': 80, 'num_leaves': 56, 'max_depth': 9, 'learning_rate': 0.07307803426594507, 'min_child_samples': 39, 'reg_lambda': 0.028370443831258373}. Best is trial 1 with value: 0.27588867665598266.
[I 2026-11-18 16:18:43,689] Trial 2 finished with value: 0.282118366665983 and parameters: {'n_estimators': 40, 'num_leaves': 52, 'max_depth': 8, 'learning_rate': 0.08890528335198394, 'min_child_samples': 26, 'reg_lambda': 6.542377205836589}. Best is trial 1 with value: 0.27588867665598266.
[I 2026-11-18 16:18:46,811] Tr

Exit: 0 Elapsed seconds: 48.0


Best TPE parameters: {'n_estimators': 110, 'num_leaves': 20, 'max_depth': 4, 'learning_rate': 0.13687258703610514, 'min_child_samples': 117, 'reg_lambda': 2.2225433823808736}


,number,value,state,params_n_estimators,params_num_leaves,params_max_depth
0,0,0.281858,COMPLETE,90,32,10
1,1,0.275889,COMPLETE,80,56,9
2,2,0.282118,COMPLETE,40,52,8
3,3,0.278161,COMPLETE,130,20,5
4,4,0.295834,COMPLETE,90,24,8
5,5,0.268281,COMPLETE,90,48,5
6,6,0.263377,COMPLETE,110,20,4
7,7,0.286661,COMPLETE,70,16,8
8,8,0.282673,COMPLETE,40,52,5
9,9,0.279284,COMPLETE,100,20,10


In [5]:
# Refit the learned scorer selected by held-out training CV. The evaluator split is never read.
review = project / 'results/review'
run('-m', 'pathsnap.reselect_release', str(context), str(review))
selection = json.loads((review / 'selection.json').read_text())
artifact = joblib.load(review / 'deployed.joblib')
assert artifact['model'] is not None
assert artifact['name'] == selection['selected'] == 'LightGBM untuned'
print('Deployed trained artifact:', artifact['name'])
print('Selection policy:', selection['selection_policy'])
print('Tuned minus untuned decoded CV interval:', selection['trusted_cv_tuned_vs_untuned'])
display(pd.read_csv(review / 'cv_summary.csv').set_index('method')[
    ['trusted_point_top1','trusted_decoded_top1','trusted_covered_log_loss']])
display(pd.read_csv(review / 'test_summary.csv').set_index('method')[
    ['point_top1','decoded_top1','route_f1','trusted_covered_log_loss']])

RUN /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/.venv/bin/python -m pathsnap.reselect_release /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/work/review-data /Users/ayushpaithankar/Documents/Codex/2026-11-18/what/outputs/pathsnap/results/review


CV-selected learned method: LightGBM untuned
{
  "selected": "LightGBM untuned",
  "trusted_cv_ranker_vs_nearest": {
    "delta": -0.004525337739656074,
    "low": -0.007298947839741666,
    "high": -0.0015297568471147381,
    "wins": 14,
    "losses": 36,
    "ties": 2
  },
  "transition_beta_m": 2.6152586684738566,
  "model_fit_trips": 78,
  "trusted_fit_observations": 180365,
  "selection_policy": "highest trusted decoded assessment CV accuracy among trained models; rules remain baselines",
  "trusted_cv_selected_vs_nearest": {
    "delta": 0.003969011815698355,
    "low": -0.0019373421712256538,
    "high": 0.012060588704532386,
    "wins": 17,
    "losses": 33,
    "ties": 2
  },
  "trusted_cv_tuned_vs_untuned": {
    "delta": -0.008494349555354429,
    "low": -0.017135542323506715,
    "high": -0.0022377906781469296,
    "wins": 22,
    "losses": 29,
    "ties": 1
  }
}



Exit: 0 Elapsed seconds: 7.2


Deployed trained artifact: LightGBM untuned
Selection policy: highest trusted decoded assessment CV accuracy among trained models; rules remain baselines
Tuned minus untuned decoded CV interval: {'delta': -0.008494349555354429, 'low': -0.017135542323506715, 'high': -0.0022377906781469296, 'wins': 22, 'losses': 29, 'ties': 1}


,trusted_point_top1,trusted_decoded_top1,trusted_covered_log_loss
method,,,
Nearest road,0.546569,0.875474,0.927972
Distance + heading,0.776996,0.861907,0.727309
LightGBM untuned,0.920354,0.879443,0.230529
LightGBM tuned without context,0.883412,0.869637,0.319147
LightGBM tuned,0.918718,0.870949,0.223183
HistGradientBoosting,0.918370,0.876338,0.233785


,point_top1,decoded_top1,route_f1,trusted_covered_log_loss
method,,,,
Nearest road,0.529489,0.885073,0.931347,0.940414
Distance + heading,0.792641,0.870198,0.936657,0.717757
LightGBM untuned,0.914196,0.885334,0.921553,0.256629
LightGBM tuned without context,0.885804,0.880585,0.928147,0.302906
LightGBM tuned,0.917380,0.878706,0.936454,0.241518
HistGradientBoosting,0.904384,0.880585,0.919374,0.267971
